In [16]:
q = 2

def key_size(m, n, w):
    return (m*m + n*w) // 8

def ct_size(m, n, w):
    return (2*m*n) // 8

def rGV_bound(q, n_code, k, m):
    """rGV для [n_code, k]_{q^m} кода."""
    w = 1
    while q_binomial(m, w, q) * q**(w*n_code) <= q**(m*(n_code - k)):
        w += 1
    return w - 1

def cpx_comb_RSD(q, n_code, k, m, w):
    """Комбинаторная атака [8] на IRSD, log2."""
    val = ((n_code - k) * m)**3 * q**(w * ceil((k+1)*m/n_code) - m)
    return int(log(max(1, val), 2))

def cpx_alg_RSD(q, n_code, k, m, w):
    """Алгебраическая атака [10] на IRSD, log2."""
    a = 0
    while m * binomial(n_code - k - 1, w) < binomial(n_code - a, w) - 1:
        a += 1
    val = q**(a*w) * m * binomial(n_code-k-1, w) * binomial(n_code-a, w)**2
    return int(log(max(1, val), 2))


def find_pir_params(L, q=2, threshold=143, max_m=30000):
    """
    Подбор минимальных параметров SHE для PIR.
    
    Фиксируем d=1 (одно умножение шифротекстов — достаточно для 2D PIR).
    
    Параметры:
      L         : число публикуемых шифротекстов клиента.
                  Для 2D PIR с N записями: L = 2*ceil(sqrt(N)).
      q         : размер базового поля (по умолчанию 2).
      threshold : порог сложности атак (143 ≈ 128 бит безопасности).
      max_m     : верхняя граница поиска по m.
    
    Возвращает (m, n, w) или None.
    """
    for m in range(L + 50, max_m):
        w_max_she = floor((sqrt(9 + 8*(m-1)) - 3) / 2)
        if w_max_she <= L:
            continue
        
        for n in range(max(20, L+1), m // 2 + 1):
            w_max_rgv = rGV_bound(q, 2*n, n, m)
            w_max = min(w_max_rgv, w_max_she)
            
            w_min = floor(L * n**2 / (n**2 + 1)) + 1
            
            if w_min > w_max:
                continue
            
            for w in range(w_min, w_max + 1):
                c1 = cpx_comb_RSD(q, n*(L+1), n, m, w)
                if c1 < threshold:
                    continue
                c2 = cpx_alg_RSD(q, n*(L+1), n, m, w)
                if c2 < threshold:
                    continue
                
                print("=" * 50)
                print(f"L = {L} публикуемых шифротекстов, q = {q}")
                print(f"  m = {m}")
                print(f"  n = {n}")
                print(f"  w = {w}")
                print("-" * 50)
                print(f"  Размер ключа:       {key_size(m, n, w)} B")
                print(f"  Размер шифротекста: {ct_size(m, n, w)} B")
                print(f"  Трафик от клиента:  {L * ct_size(m, n, w)} B")
                print(f"  cpx_comb = 2^{c1},  cpx_alg = 2^{c2}")
                print("=" * 50)
                return (m, n, w)
    
    print(f"Параметры для L={L} не найдены при m < {max_m}")
    return None


def pir_params(db_size, q=2, security=128):
    """
    Обертка для получения параметров PIR
    """
    L = 2 * ceil(sqrt(db_size))
    
    print(f"PIR конфигурация БД из {db_size} записей  =>  L = {L}")
    return find_pir_params(L, q=q, threshold=security+15)

In [17]:
find_pir_params(20)

L = 20 публикуемых шифротекстов, q = 2
  m = 351
  n = 28
  w = 25
--------------------------------------------------
  Размер ключа:       15487 B
  Размер шифротекста: 2457 B
  Трафик от клиента:  49140 B
  cpx_comb = 2^151,  cpx_alg = 2^443


(351, 28, 25)

In [18]:
pir_params(db_size=100)

PIR конфигурация БД из 100 записей  =>  L = 20
L = 20 публикуемых шифротекстов, q = 2
  m = 351
  n = 28
  w = 25
--------------------------------------------------
  Размер ключа:       15487 B
  Размер шифротекста: 2457 B
  Трафик от клиента:  49140 B
  cpx_comb = 2^151,  cpx_alg = 2^443


(351, 28, 25)

In [19]:
pir_params(db_size=10000)

PIR конфигурация БД из 10000 записей  =>  L = 200
L = 200 публикуемых шифротекстов, q = 2
  m = 20503
  n = 202
  w = 200
--------------------------------------------------
  Размер ключа:       52551676 B
  Размер шифротекста: 1035401 B
  Трафик от клиента:  207080200 B
  cpx_comb = 2^185,  cpx_alg = 2^5460


(20503, 202, 200)

In [20]:
def find_linear_pir_params(L, q=2, threshold=143):
    """
    Параметры AHE для линейного PIR с L шифротекстами в запросе.
    
    Используется только AHE (сложение + плейнтекст-умножение),
    поэтому условие SHE w(w+3)/2 + 1 < m НЕ применяется.
    
    Условия:
      1. w < m                          (корректность AHE)
      2. w ≤ rGV(q, 2n, n, m)           (единственность декодирования)
      3. L < w(1 + 1/n²)                (защита от линеаризации, Prop. 8)
      4. cpx_comb_RSD ≥ threshold       (комбинаторная атака на (L+1)-идеал. код)
      5. cpx_alg_RSD ≥ threshold        (алгебраическая атака на тот же код)
    
    Минимизируем размер шифротекста ct = 2mn/8.
    """
    L = int(L)
    q = int(q)
    best = None  # (ct_bytes, m, n, w, cpx_comb, cpx_alg)
    
    n_lo = max(L + 1, 20)
    n_hi = 4 * L + 50
    m_hi = 10 * L + 200
    
    for n in range(n_lo, n_hi + 1):
        # Для каждого n ищем минимальное m, удовлетворяющее всем условиям.
        # Минимальное m — то, при котором rGV >= w_min.
        w_min = int(L * n**2 // (n**2 + 1)) + 1
        
        for m in range(max(L + 2, n), m_hi + 1):
            w_max = min(int(rGV_bound(q, 2*n, n, m)), m - 1)
            if w_min > w_max:
                continue
            
            w = w_min
            c1 = cpx_comb_RSD(q, n*(L+1), n, m, w)
            if c1 < threshold:
                continue
            c2 = cpx_alg_RSD(q, n*(L+1), n, m, w)
            if c2 < threshold:
                continue
            
            # Нашли валидное (m, w) для этого n. Дальнейший рост m не уменьшит ct.
            ct = 2 * m * n // 8
            if best is None or ct < best[0]:
                best = (int(ct), int(m), int(n), int(w), int(c1), int(c2))
            break  # переходим к следующему n
    
    if best is None:
        print(f"Параметры для L={L} не найдены.")
        return None
    
    ct, m, n, w, c1, c2 = best
    key = (m*m + n*w) // 8
    
    # Явная конвертация Sage Rational/Integer в Python float для форматирования
    key_kb = float(key) / 1024.0
    ct_kb = float(ct) / 1024.0
    traffic_mb = float(L * ct) / 1024.0 / 1024.0
    
    print(f"=== Линейный PIR, L = N = {L}, q = {q} ===")
    print(f"  m = {m},  n = {n},  w = {w}")
    print(f"  ключ:        {key} B  ({key_kb:.1f} КБ)")
    print(f"  шифротекст:  {ct} B  ({ct_kb:.1f} КБ)")
    print(f"  трафик клиент→сервер ({L} ШТ):  {L*ct} B  ({traffic_mb:.2f} МБ)")
    print(f"  cpx_comb = 2^{c1},  cpx_alg = 2^{c2}")
    return (m, n, w)

In [21]:
find_linear_pir_params(100)

=== Линейный PIR, L = N = 100, q = 2 ===
  m = 322,  n = 182,  w = 100
  ключ:        15235 B  (14.9 КБ)
  шифротекст:  14651 B  (14.3 КБ)
  трафик клиент→сервер (100 ШТ):  1465100 B  (1.40 МБ)
  cpx_comb = 2^145,  cpx_alg = 2^2681


(322, 182, 100)

In [22]:
find_linear_pir_params(1000)

KeyboardInterrupt: 

In [ ]:
find_linear_pir_params(10)

In [ ]:
find_linear_pir_params(10000)

In [27]:
import time
from functools import lru_cache

# ---------- Размеры и тайминги ----------

def key_size(m, n, w):
    return (m*m + n*w) // 8

def ct_size(m, n, w):
    return (2*m*n) // 8

def add_time(m, n):
    t = (2*m*n) / 3000000
    return N(t) if t < 1 else floor(t)

def mul_time(m, n):
    t = (3*(m*n)**1.6) / 3000000
    return N(t) if t < 1 else floor(t)


# ---------- Мемоизированные тяжёлые функции ----------

@lru_cache(maxsize=None)
def rGV_bound_cached(q, n_code, k_code, m, w_start=1):
    w = max(1, w_start)
    threshold = q**(m*(n_code - k_code))
    while q_binomial(m, w, q) * q**(w*n_code) <= threshold:
        w += 1
    return w - 1

@lru_cache(maxsize=None)
def cpx_comb_RSD_cached(q, n_code, k_code, m, w):
    val = (((n_code - k_code)*m)**3) * (q**(w * ceil((k_code+1)*m / n_code) - m))
    return int(log(max(1, val), 2))

@lru_cache(maxsize=None)
def cpx_alg_RSD_cached(q, n_code, k_code, m, w):
    a = 0
    lhs = m * binomial(n_code - k_code - 1, w)
    while lhs < binomial(n_code - a, w) - 1:
        a += 1
    val = q**(a*w) * m * binomial(n_code - k_code - 1, w) * (binomial(n_code - a, w)**2)
    return int(log(max(1, val), 2))


# ---------- Проверка и бинпоиск ----------

def _check_n_for_m(q, m, n, w, l, security_bits):
    if l * n*n >= w * (n*n + 1):
        return (False, None, None)
    cpx_c = cpx_comb_RSD_cached(q, n*(l+1), n, m, w)
    if cpx_c < security_bits:
        return (False, cpx_c, None)
    cpx_a = cpx_alg_RSD_cached(q, n*(l+1), n, m, w)
    if cpx_a < security_bits:
        return (False, cpx_c, cpx_a)
    return (True, cpx_c, cpx_a)


def _find_min_n_for_m(q, m, l, w_min, security_bits, n_min, n_max, prev_w_guess=1):
    def good(n):
        w = rGV_bound_cached(q, 2*n, n, m, w_start=prev_w_guess)
        if w < w_min:
            return False, w, None, None
        ok, cc, ca = _check_n_for_m(q, m, n, w, l, security_bits)
        return ok, w, cc, ca

    ok_hi, w_hi, cc_hi, ca_hi = good(n_max)
    if not ok_hi:
        return None

    lo, hi = n_min, n_max
    best = (n_max, w_hi, cc_hi, ca_hi)
    while lo <= hi:
        mid = (lo + hi) // 2
        ok, w, cc, ca = good(mid)
        if ok:
            best = (mid, w, cc, ca)
            hi = mid - 1
        else:
            lo = mid + 1
    return best


# ---------- Главная функция ----------

def find_params_pir(N, pir_type, q=2, record_n_min=None, n_max_search=None,
                    security_bits=128, safety_factor=None, verbose=True,
                    m_max_search=20000, progress_every=20):
    """Поиск параметров для PIR из ноутбука: 'obvious' / 'pir' / 'tensor'.

    pir_type:
        'obvious' — ObviousPIRClient: ℓ = N,        AHE
        'pir'     — PIRClient (sqrt): ℓ = ⌈√N⌉,    AHE
        'tensor'  — TensorPIRClient:  ℓ = 2⌈√N⌉,   SHE
    """
    t0 = time.time()

    if pir_type == 'obvious':
        l = N
        needs_she = False
    elif pir_type == 'pir':
        l = int(ceil(sqrt(N)))
        needs_she = False
    elif pir_type == 'tensor':
        l = 2 * int(ceil(sqrt(N)))
        needs_she = True
    else:
        raise ValueError(f'Unknown pir_type: {pir_type}')

    # Надёжная конструкция 4/3 в Sage:
    sf = Integer(4)/Integer(3) if safety_factor is None else Rational(safety_factor)
    w_min = int(ceil(sf * l))

    if needs_she:
        m_low_constraint = w_min*(w_min + 3)//2 + 2
    else:
        m_low_constraint = w_min + 1
    m_low = max(m_low_constraint, 3*w_min)

    n_min_user = record_n_min if record_n_min is not None else max(40, w_min + 5)
    n_max_user = n_max_search if n_max_search is not None else max(n_min_user*3, w_min*3)

    if verbose:
        scheme = 'SHE' if needs_she else 'AHE'
        print('===== PIR Parameter Search =====')
        print(f'  N (db size)         : {N}')
        print(f'  PIR type            : {pir_type}  ({scheme})')
        print(f'  ciphertexts (ℓ)     : {l}')
        print(f'  required w ≥        : {w_min}  (safety = {sf})')
        print(f'  m search starts at  : {m_low}')
        print(f'  n search in         : [{n_min_user}, {n_max_user}]')
        print(f'  security target     : {security_bits} bits')
        print()

    prev_w = 1
    checked = 0
    for m in range(m_low, m_low + m_max_search):
        n_max = min(m // 2, n_max_user)
        if n_max < n_min_user:
            checked += 1
            continue

        result = _find_min_n_for_m(q, m, l, w_min, security_bits, n_min_user, n_max,
                                   prev_w_guess=prev_w)
        checked += 1
        if result is not None:
            n_found, w, cc, ca = result
            elapsed = time.time() - t0
            res = {
                'q': q, 'm': m, 'n': n_found, 'w': w, 'l': l,
                'pir_type': pir_type,
                'scheme': 'SHE' if needs_she else 'AHE',
                'comb_security': cc, 'alg_security': ca,
                'security': min(cc, ca),
                'key_size_B': key_size(m, n_found, w),
                'ct_size_B':  ct_size(m, n_found, w),
                'add_time_ms': add_time(m, n_found),
                'search_time_s': elapsed,
            }
            if needs_she:
                res['mul_time_ms'] = mul_time(m, n_found)

            if verbose:
                print(f'FOUND in {elapsed:.1f}s (checked {checked} values of m)')
                print(f'  m = {m},  n = {n_found},  w = {w}')
                print(f'  comb attack: {cc} bits   |   alg attack: {ca} bits')
                print(f'  key size: {res["key_size_B"]} B')
                print(f'  ct  size: {res["ct_size_B"]} B')
                print(f'  add:      {res["add_time_ms"]} ms')
                if needs_she:
                    print(f'  mul_ct:   {res["mul_time_ms"]} ms')
            return res

        prev_w = max(1, w_min // 2)

        if verbose and progress_every and checked % progress_every == 0:
            elapsed = time.time() - t0
            print(f'  ... m={m}, checked={checked}, elapsed={elapsed:.1f}s')

    if verbose:
        print(f'No suitable parameters found in {time.time()-t0:.1f}s.')
    return None

In [28]:
find_params_pir(N=100, pir_type='obvious')   # AHE
find_params_pir(N=1000, pir_type='obvious')  # AHE — уже тяжело
find_params_pir(N=10000, pir_type='pir')     # то же самое по что N=100 obvious
find_params_pir(N=100, pir_type='tensor')    # SHE -> большое m

===== PIR Parameter Search =====
  N (db size)         : 100
  PIR type            : obvious  (AHE)
  ciphertexts (ℓ)     : 100
  required w ≥        : 134  (safety = 4/3)
  m search starts at  : 402
  n search in         : [139, 417]
  security target     : 128 bits

  ... m=421, checked=20, elapsed=0.3s
  ... m=441, checked=40, elapsed=0.5s
FOUND in 0.9s (checked 57 values of m)
  m = 458,  n = 229,  w = 134
  comb attack: 281 bits   |   alg attack: 3558 bits
  key size: 30056 B
  ct  size: 26220 B
  add:      0.0699213333333333 ms
===== PIR Parameter Search =====
  N (db size)         : 1000
  PIR type            : obvious  (AHE)
  ciphertexts (ℓ)     : 1000
  required w ≥        : 1334  (safety = 4/3)
  m search starts at  : 4002
  n search in         : [1339, 4017]
  security target     : 128 bits

  ... m=4021, checked=20, elapsed=647.9s
  ... m=4041, checked=40, elapsed=1292.2s
  ... m=4061, checked=60, elapsed=1953.5s
  ... m=4081, checked=80, elapsed=2623.3s
  ... m=4101, chec

{'q': 2,
 'm': 407,
 'n': 40,
 'w': 36,
 'l': 20,
 'pir_type': 'tensor',
 'scheme': 'SHE',
 'comb_security': 367,
 'alg_security': 637,
 'security': 367,
 'key_size_B': 20886,
 'ct_size_B': 4070,
 'add_time_ms': 0.0108533333333333,
 'search_time_s': 0.017879009246826172,
 'mul_time_ms': 5}

In [29]:
find_params_pir(N=10000, pir_type='tensor')

===== PIR Parameter Search =====
  N (db size)         : 10000
  PIR type            : tensor  (SHE)
  ciphertexts (ℓ)     : 200
  required w ≥        : 267  (safety = 4/3)
  m search starts at  : 36047
  n search in         : [272, 816]
  security target     : 128 bits

FOUND in 515.2s (checked 1 values of m)
  m = 36047,  n = 272,  w = 269
  comb attack: 12465 bits   |   alg attack: 7345 bits
  key size: 162432422 B
  ct  size: 2451196 B
  add:      6 ms
  mul_ct:   153568 ms


{'q': 2,
 'm': 36047,
 'n': 272,
 'w': 269,
 'l': 200,
 'pir_type': 'tensor',
 'scheme': 'SHE',
 'comb_security': 12465,
 'alg_security': 7345,
 'security': 7345,
 'key_size_B': 162432422,
 'ct_size_B': 2451196,
 'add_time_ms': 6,
 'search_time_s': 515.1997690200806,
 'mul_time_ms': 153568}